# 3일차 파트 1 실습 풀이

`3일차_실습.ipynb`의 풀이와 해설입니다. 풀이는 하나의 예시이며, 테스트 셀을 통과한다면 다르게 작성해도 됩니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: python -m pip install sentence-transformers scikit-learn pypdf python-docx openpyxl pandas
import numpy as np

import embed

DATA = 'data/3일차'
CORPUS = 'data/2일차/corpus'      # 2일차와 같은 문서 253건
OUT = 'out'                        # 예제가 만드는 파일을 두는 폴더
os.makedirs(OUT, exist_ok=True)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 준비

아래 셀은 실습이 쓰는 재료와 클래스를 만듭니다. 그대로 실행합니다. 문서 253건의 임베딩을 계산하므로 GPU가 없는 PC에서는 수십 초가 걸리기도 합니다(교재를 만든 PC에서 같은 계산을 한 예제 1-1은 24초였습니다. 값은 PC와 실행할 때마다 다릅니다). 이 실습에는 `numpy`, `sentence-transformers`와 임베딩 모델이 필요합니다. 두 패키지는 위 "준비 (2)" 셀 안의 `# 설치:` 주석에 있는 명령으로 설치합니다. 임베딩 모델 준비는 교재 3일차의 "실습 환경 준비"를 따릅니다.

| 이름 | 내용 |
| --- | --- |
| `VectorIndex` | 교재 예제 1-2의 클래스입니다. 벡터, 문서 번호, 본문을 한 객체에 두고 `replace`, `add`, `remove`, `search`, `rank_of`를 제공합니다 |
| `BASE` | 변경을 반영하기 전의 인덱스(문서 253건)입니다. 실습에서는 복사본에 반영해 `BASE`를 그대로 둡니다 |
| `embed_text(text)` | 글 한 건을 벡터로 바꿉니다 |
| `report(index, label)` | 질문 6개의 정답 순위와 1위 문서를 출력합니다 |

In [ ]:
# 실습 준비
import copy

import checks_d3

class VectorIndex:
    """벡터, 문서 번호, 본문을 한 객체에 두고 항상 같은 순서로 고칩니다."""

    def __init__(self, names, docs, vectors, model):
        if not len(names) == len(docs) == len(vectors):
            raise ValueError(f'개수가 다릅니다: 문서 번호 {len(names)}, 본문 {len(docs)}, 벡터 {len(vectors)}')
        self.names, self.docs, self.model = list(names), list(docs), model
        self.vectors = np.array(vectors)

    def __len__(self):
        return len(self.names)

    def search(self, query_vector, k=3):
        """질문 벡터와 가까운 상위 k건의 (문서 번호, 점수)를 돌려줍니다."""
        scores = self.vectors @ query_vector
        order = np.argsort(-scores, kind='stable')[:k]
        return [(self.names[row], round(float(scores[row]), 3)) for row in order]

    def rank_of(self, query_vector, gold):
        """정답 문서 gold가 몇 위인지 돌려줍니다. 1위가 1입니다."""
        order = list(np.argsort(-(self.vectors @ query_vector), kind='stable'))
        return order.index(self.names.index(gold)) + 1

    def replace(self, name, doc, vector):
        """개정: 같은 행의 본문과 벡터를 바꿉니다."""
        row = self.names.index(name)
        self.docs[row], self.vectors[row] = doc, vector

    def add(self, name, doc, vector):
        """신규: 맨 아래에 행을 추가합니다."""
        if name in self.names:
            raise ValueError(f'{name}은 이미 있습니다. 고치려면 replace()를 호출합니다')
        self.names.append(name)
        self.docs.append(doc)
        self.vectors = np.vstack([self.vectors, vector])

    def remove(self, name):
        """폐기: 그 문서의 행을 지웁니다."""
        row = self.names.index(name)
        del self.names[row], self.docs[row]
        self.vectors = np.delete(self.vectors, row, axis=0)

    def save(self, folder):
        np.save(f'{folder}/index.npy', self.vectors)
        with open(f'{folder}/index.json', 'w', encoding='utf-8') as file:
            json.dump({'model': self.model, 'names': self.names, 'docs': self.docs}, file, ensure_ascii=False)

    @classmethod
    def load(cls, folder, model=None):
        """저장한 인덱스를 불러옵니다. model을 주면 저장된 모델 이름과 같은지 확인하고, 다르면 ValueError를 냅니다."""
        with open(f'{folder}/index.json', encoding='utf-8') as file:
            saved = json.load(file)
        if model is not None and saved['model'] != model:
            raise ValueError(f'이 인덱스는 {saved["model"]}로 만들었는데 지금 모델은 {model}입니다')
        return cls(saved['names'], saved['docs'], np.load(f'{folder}/index.npy'), saved['model'])

names = sorted(file[:-3] for file in os.listdir(CORPUS) if file.endswith('.md'))
docs = [open(f'{CORPUS}/{name}.md', encoding='utf-8').read() for name in names]
BASE = VectorIndex(names, docs, embed.passages(docs), embed.model_name())


def embed_text(text):
    return embed.passages([text])[0]


questions = pd.read_csv(f'{DATA}/변경_질문_6.csv', encoding='utf-8-sig', keep_default_na=False)


def report(index, label):
    print(f'{label}, 행 수 {len(index)}')
    for question, gold in zip(questions['질문'], questions['정답']):
        vector = embed.query(question)
        top = index.search(vector, 1)[0][0]
        place = f'정답 {index.rank_of(vector, gold)}위' if gold in index.names else '정답 문서 없음'
        print(f'   {question[:26]:28s} {place}, 1위 {top}')


print(f'문서 {len(BASE):,}건, 인덱스 {BASE.vectors.shape}')

## 실습 1 변경 목록을 인덱스에 반영하기

**목적** 문서의 개정, 추가, 폐기를 바뀐 행만 고쳐 반영하는 코드를 직접 작성합니다. 변경의 일부를 빠뜨리면 검색 결과가 어떻게 달라지는지 확인합니다.

**재료** `data/3일차/변경_목록.csv`(변경 아홉 건), `data/3일차/변경_문서/`(개정본과 신규 문서 4건), `data/3일차/변경_질문_6.csv`(질문 6개와 정답 문서).

**할 일** `apply_changes(index, revised, added, retired, folder, embed_text)`를 작성합니다. `index`는 `VectorIndex`이고, `revised`, `added`, `retired`는 개정, 신규, 폐기할 문서 번호의 목록입니다. 개정본과 신규 문서는 `folder`에 `문서번호.md` 파일로 있고, `embed_text(글)`은 글 한 건의 벡터를 돌려줍니다.

1. 고치기 전에 목록을 검사합니다. 개정 또는 폐기할 문서가 인덱스에 없거나, 신규로 넣을 문서가 이미 있거나, 개정과 신규의 문서 파일이 없으면 `ValueError`를 냅니다. 이때 인덱스는 아무것도 고치지 않은 상태여야 합니다.
2. 개정과 신규 문서의 벡터는 인덱스를 고치기 전에 모두 계산합니다. 임베딩 중에 오류가 나도 인덱스는 아무것도 바뀌지 않아야 합니다.
3. 개정은 `replace`, 신규는 `add`, 폐기는 `remove`로 반영합니다.
4. 개정 건수, 신규 건수, 폐기 건수, 반영 후 행 수를 `{"교체": 2, "추가": 2, "삭제": 5, "행 수": 250}` 꼴의 딕셔너리(`dict`)로 돌려줍니다. 키 이름은 이대로 씁니다(교체는 개정, 추가는 신규, 삭제는 폐기의 건수입니다).

**완료 기준** 테스트 셀이 "실습 1 통과"를 출력합니다. 반영 후 인덱스는 250행이고 벡터, 문서 번호, 본문의 개수가 같습니다. 그다음 실제 문서에 반영하는 셀과 빠뜨린 두 경우의 셀을 실행해, 아래 "적어 보기"를 채웁니다.

<details><summary>힌트 1</summary>

검사와 반영을 같은 반복문에서 하면 목록의 뒤쪽에 틀린 문서가 있을 때 앞쪽은 이미 고쳐진 뒤입니다. 검사를 모두 끝낸 뒤에 반영을 시작합니다.
</details>

<details><summary>힌트 2</summary>

파일 존재는 `os.path.exists(f'{folder}/{name}.md')`로 확인합니다. 인덱스에 있는지는 `name in index.names`입니다. `add`는 맨 아래에 행을 붙이고, 이미 있는 이름이면 `ValueError`를 냅니다.
</details>

<details><summary>힌트 3</summary>

벡터는 반영하기 전에 `{문서 번호: 벡터}` 딕셔너리로 모두 만들어 둡니다. 반영 단계에서는 이 딕셔너리만 읽습니다.
</details>

<details><summary>힌트 4</summary>

개정본의 본문은 `open(f'{folder}/{name}.md', encoding='utf-8').read()`로 읽고, 그 글을 `embed_text()`에 넣어 벡터를 받아 `index.replace(name, doc, vector)`에 넘깁니다.
</details>

In [ ]:
# 실습 1
def apply_changes(index, revised, added, retired, folder, embed_text):
    """변경 목록을 인덱스에 반영하고 건수를 돌려줍니다. 목록이 틀렸으면 ValueError를 내고 아무것도 고치지 않습니다."""
    # 반영하기 전에 모두 검사합니다. 하나라도 틀리면 인덱스를 건드리지 않습니다
    missing = [name for name in revised + retired if name not in index.names]
    existing = [name for name in added if name in index.names]
    no_file = [name for name in revised + added if not os.path.exists(f'{folder}/{name}.md')]
    if missing or existing or no_file:
        raise ValueError(f'인덱스에 없는 문서 {missing}, 이미 있는 문서 {existing}, 파일이 없는 문서 {no_file}')

    # 개정과 신규 문서의 벡터를 먼저 모두 계산합니다. 임베딩 중에 오류가 나도 인덱스는 그대로입니다
    docs_of = {name: open(f'{folder}/{name}.md', encoding='utf-8').read() for name in revised + added}
    vectors_of = {name: embed_text(doc) for name, doc in docs_of.items()}

    for name in revised:
        index.replace(name, docs_of[name], vectors_of[name])
    for name in added:
        index.add(name, docs_of[name], vectors_of[name])
    for name in retired:
        index.remove(name)
    return {'교체': len(revised), '추가': len(added), '삭제': len(retired), '행 수': len(index)}

In [ ]:
# 실습 1 확인 (임베딩 모델도 API도 부르지 않습니다)
checks_d3.check_apply_changes(apply_changes, VectorIndex)

In [ ]:
# 실습 1 (1) 반영하기 전과 전체를 반영한 뒤
changes = pd.read_csv(f'{DATA}/변경_목록.csv', encoding='utf-8-sig')
REVISED = list(changes[changes['종류'] == '개정']['문서'])
ADDED = list(changes[changes['종류'] == '신규']['문서'])
RETIRED = list(changes[changes['종류'] == '폐기']['문서'])


def reflect(revised, added, retired):
    """BASE의 복사본에 변경을 반영해 돌려줍니다. BASE는 그대로 둡니다."""
    trial = copy.deepcopy(BASE)
    result = apply_changes(trial, revised, added, retired, f'{DATA}/변경_문서', embed_text)
    print(result)
    return trial


report(BASE, '반영 전')
full = reflect(REVISED, ADDED, RETIRED)
report(full, '전체 반영')

In [ ]:
# 실습 1 (2) 일부를 빠뜨리면
# 경우 A: EHS-001의 개정을 빠뜨립니다
missed_revision = reflect(['SOP-ETCH-005'], ADDED, RETIRED)
vector = embed.query(questions['질문'][1])
top_name = missed_revision.search(vector, 1)[0][0]
print(f"경우 A, 질문 2의 1위 {top_name}, EHS-001의 순위 {missed_revision.rank_of(vector, 'EHS-001')}")
print('   1위 문서에서 "감지기"가 든 줄:',
      [line for line in missed_revision.docs[missed_revision.names.index(top_name)].split('\n') if '감지기' in line])

# 경우 B: 폐기 다섯 건 가운데 두 건만 지웁니다
missed_retired = reflect(REVISED, ADDED, ['NOTI-001', 'NOTI-007'])
top3 = missed_retired.search(embed.query(questions['질문'][5]), 3)
print('경우 B, 질문 6의 상위 3건', top3)
print('   그 문서들의 첫 줄:', [missed_retired.docs[missed_retired.names.index(name)].splitlines()[0] for name, _ in top3])

**해설**

아래 숫자는 2026-11-24에 실행한 결과입니다. 같은 임베딩 모델을 쓰면 거의 같은 값이 나옵니다.

- 전체를 반영하면 인덱스는 253 + 2 - 5 = 250행이고, 반환값은 `{'교체': 2, '추가': 2, '삭제': 5, '행 수': 250}`입니다. 정답 문서가 있는 질문 5개는 모두 정답 문서가 1위가 됩니다. 정답이 없는 질문 6의 1위는 셧다운과 관계없는 `EHS-020`입니다. 폐기한 공지는 나오지 않지만, 검색은 답이 없어도 항상 어떤 문서를 돌려줍니다. 그래서 근거가 없을 때 모델이 답을 거절하게 하는 규칙이 필요합니다(파트 3). 반영하기 전에는 질문 2의 정답 `EHS-001`이 4위였고, 질문 3과 4의 정답 문서는 인덱스에 없었습니다.
- **경우 A**: 질문 2의 1위는 `EHS-016`이고 `EHS-001`은 4위입니다. 상위 4건은 점수가 0.885~0.886으로 거의 같아서 PC에 따라 순서가 바뀔 수 있지만, 개정 전의 `EHS-001`이 1위가 아니라는 점은 같습니다. `EHS-016` 본문에 "감지기"가 든 줄은 없습니다. 이 인덱스로 RAG를 만들면 15분 모니터링 절이 든 개정본이 인덱스에 없으므로, 모델은 그 절을 근거로 답할 수 없습니다. 모델은 근거를 찾지 못했다고 답하거나, 개정 전 문서의 내용으로 답할 수 있습니다. 현장 작업자는 개정된 안전 절차를 받지 못합니다. 응답에 오류가 나지 않으므로 인덱스가 오래됐다는 것을 응답만 봐서는 알 수 없습니다.
- **경우 B**: 폐기를 두 건만 지우면 인덱스는 253행으로 그대로이고, 질문 6의 상위 3건이 모두 지난 셧다운 공지(`NOTI-013`, `NOTI-019`, `NOTI-025`)입니다. 반환값의 `'삭제'`는 인자로 넘긴 목록의 길이일 뿐이어서, 폐기가 빠진 것은 `'행 수'`가 253이라는 것으로 알 수 있습니다. 변경 목록의 건수에서 기대하는 행 수(253 + 2 - 5 = 250)를 계산해 두고 반환값의 행 수와 비교하면 반영을 놓친 변경을 바로 찾을 수 있습니다.
- `apply_changes()`가 고치기 전에 목록을 모두 검사하는 이유는, 목록의 뒤쪽에 틀린 문서가 있을 때 앞쪽만 반영된 인덱스가 남지 않게 하기 위해서입니다. 반만 반영된 인덱스는 경우 A와 B처럼 응답에 오류 없이 옛 내용이 섞입니다.